# Tutorial 3: Production Rules and Procedural Memory

Production rules are ACT-R's core mechanism for modeling procedural knowledge - the knowledge of how to do things. While we've used simple productions in earlier tutorials, this one digs into their full capabilities: pattern matching, conflict resolution, utility learning, and how they model the transition from novice to expert performance.

## Table of Contents
1. [Production Rules: The Engine of Cognition](#production-rules)
2. [Buffers and Pattern Matching](#buffers-matching)
3. [Conflict Resolution](#conflict-resolution)
4. [Procedural Learning](#procedural-learning)
5. [Real-World Application: Skill Acquisition](#skill-acquisition)
6. [Exercises](#exercises)

## Introduction

In Tutorials 1 and 2, we used simple production rules without examining them closely. But productions are ACT-R's model of procedural knowledge - knowing how rather than knowing that.

The key insight: humans don't consciously think through every step of skilled behavior. Expertise involves developing efficient procedures that become automatic through practice. This tutorial shows you how to model that transition from novice to expert.

In [ ]:
import pyactr as actr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from collections import defaultdict

np.random.seed(42)

## Production Rules: The Engine of Cognition <a id='production-rules'></a>

### Production Anatomy

A production has two parts:
- LHS (Left-Hand Side): Conditions that must be met
- RHS (Right-Hand Side): Actions to perform

```
production_name:
    IF (conditions)     # LHS
    THEN (actions)      # RHS
```

### Buffer Prefixes

- `=buffer>`: Match the contents of a buffer
- `?buffer>`: Query the state of a buffer
- `+buffer>`: Make a request to a module
- `~buffer>`: Clear a buffer

In [ ]:
prod_model = actr.ACTRModel()

actr.chunktype("arithmetic", ("operation", "arg1", "arg2", "result"))
actr.chunktype("goal", ("state", "task", "arg1", "arg2", "result"))

# Example 1: Simple addition production
prod_model.productionstring(name="add_single_digit", string="""
    =g>
        isa      goal
        state    calculate
        task     add
        arg1     =x
        arg2     =y
        result   None
    ==>
    =g>
        result   7
        state    done
""")

# Example 2: Production with retrieval
prod_model.productionstring(name="retrieve_addition_fact", string="""
    =g>
        isa      goal
        state    calculate
        task     add
        arg1     =x
        arg2     =y
        result   None
    ?retrieval>
        state    free
    ==>
    =g>
        state    retrieving
    +retrieval>
        isa      arithmetic
        operation add
        arg1     =x
        arg2     =y
""")

print("Production rules defined!")
print("\nPattern matching with variables (=x, =y)")
print("Buffer queries (?retrieval>)")
print("Module requests (+retrieval>)")
print("State transitions (calculate -> retrieving)")

### Production Features

Productions can include:
- Variable binding: `=x` binds a value to variable x
- Variable matching: Use the same variable in multiple places
- Negation: `~=x` means "not equal to x"
- Wildcards: `*` matches anything

In [ ]:
# Example 3: Production with negation and multiple conditions
prod_model.productionstring(name="handle_complex_calculation", string="""
    =g>
        isa      goal
        state    calculate
        task     =task
        arg1     =x
        arg2     =y
        result   None
    ?retrieval>
        state    free
    ?imaginal>
        state    free
    ==>
    =g>
        state    working
    +imaginal>
        isa      arithmetic
        operation =task
        arg1     =x
        arg2     =y
""")

# Example 4: Error handling with buffer clearing
prod_model.productionstring(name="general_error_handler", string="""
    =g>
        isa      goal
        state    =state
        state    ~done
        state    ~start
    ?retrieval>
        state    error
    ==>
    =g>
        state    error_recovery
    ~retrieval>
""")

print("Advanced features:")
print("1. Multiple buffer queries in one production")
print("2. Negation constraints (state ~done)")
print("3. Buffer clearing (~retrieval>)")
print("4. Error handling patterns")

## Buffers and Pattern Matching <a id='buffers-matching'></a>

ACT-R has several buffers that productions can access:

- Goal Buffer: Current goal/intention
- Retrieval Buffer: Retrieved memory
- Imaginal Buffer: Mental imagery/working memory
- Visual Buffer: Visual attention
- Manual Buffer: Motor actions

In [ ]:
multi_buffer_model = actr.ACTRModel()

actr.chunktype("visual_object", ("color", "shape", "size"))
actr.chunktype("classification", ("color", "shape", "category"))
actr.chunktype("task", ("state", "color", "shape", "category"))

dm = multi_buffer_model.decmem

classifications = [
    {"color": "red", "shape": "circle", "category": "target"},
    {"color": "blue", "shape": "square", "category": "distractor"},
    {"color": "green", "shape": "triangle", "category": "distractor"},
    {"color": "red", "shape": "square", "category": "maybe-target"},
]

for item in classifications:
    dm.add(actr.makechunk(typename="classification", **item))

# Transfer visual information to goal buffer
multi_buffer_model.productionstring(name="classify_visual_object", string="""
    =g>
        isa      task
        state    attending
        color    None
        shape    None
    =visual>
        isa      visual_object
        color    =color
        shape    =shape
    ==>
    =g>
        state    classifying
        color    =color
        shape    =shape
    +retrieval>
        isa      classification
        color    =color
        shape    =shape
""")

# Use retrieval result
multi_buffer_model.productionstring(name="report_classification", string="""
    =g>
        isa      task
        state    classifying
        color    =col
        shape    =shp
    =retrieval>
        isa      classification
        color    =col
        shape    =shp
        category =cat
    ==>
    =g>
        state    done
        category =cat
""")

print("Multi-buffer coordination example created!")

## Conflict Resolution <a id='conflict-resolution'></a>

When multiple productions match, ACT-R must choose which one to fire. This is conflict resolution.

Selection criteria:
1. Utility: Productions can have utility values
2. Specificity: More specific productions are preferred
3. Recency: Recently used productions may be preferred

In [ ]:
conflict_model = actr.ACTRModel()

conflict_model.model_parameters["subsymbolic"] = True
conflict_model.model_parameters["utility_noise"] = 0.1

actr.chunktype("strategy_goal", ("state", "number", "strategy"))

# Two competing strategies for the same situation
# Strategy 1: Count up (novice strategy)
conflict_model.productionstring(name="count_up_strategy", 
                               string="""
    =g>
        isa      strategy_goal
        state    choose
        number   =n
    ==>
    =g>
        state    execute
        strategy count-up
""", 
                               utility=1)

# Strategy 2: Direct retrieval (expert strategy)  
conflict_model.productionstring(name="retrieval_strategy",
                               string="""
    =g>
        isa      strategy_goal
        state    choose
        number   =n
    ==>
    =g>
        state    execute
        strategy direct-retrieval
""",
                               utility=5)

# More specific production (wins over general ones)
conflict_model.productionstring(name="special_case_seven",
                               string="""
    =g>
        isa      strategy_goal
        state    choose
        number   7
    ==>
    =g>
        state    execute
        strategy special-seven
""",
                               utility=3)

print("Conflict resolution demonstration:")
print("\nThree productions compete:")
print("1. count_up_strategy (utility=1) - General, low utility")
print("2. retrieval_strategy (utility=5) - General, high utility") 
print("3. special_case_seven (utility=3) - Specific to number 7")

# Run multiple simulations to see conflict resolution
results = defaultdict(int)
for i in range(20):
    conflict_model.goal.clear()
    conflict_model.goal.add(actr.makechunk(typename="strategy_goal", 
                                          state="choose", 
                                          number=7))
    
    sim = conflict_model.simulation(gui=False, trace=False)
    sim.run(0.1)
    
    goal_chunks = list(conflict_model.goal)
    if goal_chunks and hasattr(goal_chunks[0], 'strategy'):
        results[goal_chunks[0].strategy] += 1

print("\nResults from 20 runs with number=7:")
for strategy, count in results.items():
    print(f"  {strategy}: {count} times ({count/20*100:.0f}%)")

## Procedural Learning <a id='procedural-learning'></a>

ACT-R models how skills improve with practice through:

1. Utility Learning: Productions that lead to success get higher utilities
2. Production Compilation: Combining productions for efficiency
3. Speedup: Productions fire faster with practice

In [ ]:
learning_model = actr.ACTRModel()

learning_model.model_parameters["subsymbolic"] = True
learning_model.model_parameters["utility_learning"] = True
learning_model.model_parameters["utility_alpha"] = 0.2
learning_model.model_parameters["utility_noise"] = 0.5

actr.chunktype("math_goal", ("state", "problem", "answer", "method"))
actr.chunktype("math_fact", ("problem", "answer"))

dm = learning_model.decmem
math_facts = [
    ("3+4", "7"), ("5+2", "7"), ("6+3", "9"),
    ("2+2", "4"), ("4+5", "9"), ("7+1", "8")
]

for problem, answer in math_facts:
    dm.add(actr.makechunk(typename="math_fact", 
                         problem=problem, 
                         answer=answer))

# Strategy 1: Guess (always guesses 7)
learning_model.productionstring(name="guess_strategy",
                               string="""
    =g>
        isa      math_goal
        state    solve
        problem  =prob
        answer   None
    ==>
    =g>
        method   guess
        answer   7
        state    check
""",
                               utility=0,
                               reward=1)

# Strategy 2: Retrieve from memory
learning_model.productionstring(name="retrieve_strategy_init",
                               string="""
    =g>
        isa      math_goal
        state    solve
        problem  =prob
        answer   None
    ==>
    =g>
        state    retrieving
        method   memory
    +retrieval>
        isa      math_fact
        problem  =prob
""",
                               utility=0,
                               reward=2)

learning_model.productionstring(name="retrieve_answer",
                               string="""
    =g>
        isa      math_goal
        state    retrieving
        problem  =prob
    =retrieval>
        isa      math_fact
        problem  =prob
        answer   =ans
    ==>
    =g>
        answer   =ans
        state    check
""")

print("Learning model with two strategies:")
print("1. Guessing (fast but often wrong)")
print("2. Memory retrieval (slower but accurate)")
print("\nWith utility learning, successful strategies increase in utility over time.")

problems = ["3+4", "5+2", "6+3", "2+2", "3+4", "5+2"]
strategy_use = defaultdict(int)

print("\nSimulating 6 math problems...")
for i, problem in enumerate(problems):
    learning_model.goal.clear()
    learning_model.goal.add(actr.makechunk(typename="math_goal",
                                         state="solve",
                                         problem=problem))
    
    sim = learning_model.simulation(gui=False, trace=False)
    sim.run(2)
    
    goal_chunks = list(learning_model.goal)
    if goal_chunks and hasattr(goal_chunks[0], 'method'):
        method = goal_chunks[0].method
        strategy_use[method] += 1
        
        if hasattr(goal_chunks[0], 'answer'):
            answer = goal_chunks[0].answer
            correct = (problem == "3+4" and answer == "7") or \
                     (problem == "5+2" and answer == "7") or \
                     (problem != "3+4" and problem != "5+2" and answer != "7")
            
            print(f"Problem {i+1}: {problem} → Method: {method}, "
                  f"Answer: {answer}, Correct: {correct}")

print("\nStrategy use summary:")
for method, count in strategy_use.items():
    print(f"  {method}: {count} times")

## Real-World Application: Skill Acquisition <a id='skill-acquisition'></a>

Here's a model of how someone learns to type - transitioning from hunt-and-peck to touch typing. This demonstrates multiple strategies competing, skill improvement over time, and the development of automaticity.

In [ ]:
typing_model = actr.ACTRModel()

typing_model.model_parameters["subsymbolic"] = True
typing_model.model_parameters["utility_noise"] = 0.3

actr.chunktype("typing_goal", ("state", "letter", "method"))
actr.chunktype("key_location", ("letter", "finger", "row", "position"))

dm = typing_model.decmem

# Simplified keyboard layout (home row)
key_knowledge = [
    ("a", "left-pinky", "home", 1),
    ("s", "left-ring", "home", 2),
    ("d", "left-middle", "home", 3),
    ("f", "left-index", "home", 4),
    ("j", "right-index", "home", 5),
    ("k", "right-middle", "home", 6),
    ("l", "right-ring", "home", 7),
]

for letter, finger, row, pos in key_knowledge:
    dm.add(actr.makechunk(typename="key_location",
                         letter=letter,
                         finger=finger,
                         row=row,
                         position=pos))

# Strategy 1: Visual search (hunt and peck)
typing_model.productionstring(name="hunt_and_peck",
                             string="""
    =g>
        isa      typing_goal
        state    type
        letter   =letter
        method   None
    ==>
    =g>
        state    searching
        method   visual-search
""",
                             utility=5)

# Strategy 2: Memory retrieval
typing_model.productionstring(name="memory_typing",
                             string="""
    =g>
        isa      typing_goal
        state    type
        letter   =letter
        method   None
    ==>
    =g>
        state    recalling
        method   memory
    +retrieval>
        isa      key_location
        letter   =letter
""",
                             utility=2)

typing_model.productionstring(name="execute_from_memory",
                             string="""
    =g>
        isa      typing_goal
        state    recalling
        letter   =letter
    =retrieval>
        isa      key_location
        letter   =letter
        finger   =finger
    ==>
    =g>
        state    done
""")

# Strategy 3: Direct motor execution (touch typing)
# Only works for familiar keys
typing_model.productionstring(name="touch_typing",
                             string="""
    =g>
        isa      typing_goal
        state    type
        letter   =letter
        method   None
        letter   ~x
        letter   ~q
    ==>
    =g>
        state    done
        method   automatic
""",
                             utility=0)

print("Typing skill acquisition model")
print("\nThree strategies compete:")
print("1. Hunt-and-peck - Visual search, initially preferred")
print("2. Memory retrieval - Recall key location")
print("3. Touch typing - Automatic motor program")

letters_to_type = ['a', 's', 'd', 'f', 'a', 's', 'j', 'k', 'a', 'd']
methods_used = []

method_times = {
    'visual-search': 2.0,
    'memory': 0.5,
    'automatic': 0.2,
    None: 1.0
}

print("\nSimulating typing 10 letters:")
for i, letter in enumerate(letters_to_type):
    typing_model.goal.clear()
    typing_model.goal.add(actr.makechunk(typename="typing_goal",
                                       state="type",
                                       letter=letter))
    
    sim = typing_model.simulation(gui=False, trace=False)
    sim.run(3)
    
    goal_chunks = list(typing_model.goal)
    if goal_chunks:
        chunk = goal_chunks[0]
        method = chunk.method if hasattr(chunk, 'method') else None
        methods_used.append(method)
        
        typing_time = method_times.get(method, 1.0)
        
        print(f"Letter {i+1} '{letter}': {method} ({typing_time}s)")
        
        if method == "automatic" and i < 5:
            typing_model.productions["touch_typing"].utility += 0.5

if methods_used:
    method_counts = {}
    for method in methods_used:
        method_counts[method] = method_counts.get(method, 0) + 1
    
    print("\nStrategy usage summary:")
    for method, count in method_counts.items():
        print(f"  {method}: {count} times ({count/len(methods_used)*100:.0f}%)")
    
    typing_times = [method_times.get(m, 1.0) for m in methods_used]
    print(f"\nAverage typing time: {np.mean(typing_times):.2f} seconds")
    
    plt.figure(figsize=(10, 6))
    
    plt.subplot(2, 1, 1)
    plt.plot(range(1, len(typing_times)+1), typing_times, 'bo-', linewidth=2)
    plt.xlabel('Letter Number')
    plt.ylabel('Typing Time (seconds)')
    plt.title('Typing Speed Over Time')
    plt.ylim(0, 2.5)
    plt.grid(True, alpha=0.3)
    
    for i, (time, method) in enumerate(zip(typing_times, methods_used)):
        if method:
            plt.annotate(method, (i+1, time), 
                        textcoords="offset points", 
                        xytext=(0,10), ha='center', fontsize=8, rotation=45)
    
    plt.subplot(2, 1, 2)
    strategy_colors = {'visual-search': 'red', 'memory': 'yellow', 'automatic': 'green'}
    for i, method in enumerate(methods_used):
        if method:
            plt.bar(i+1, 1, color=strategy_colors.get(method, 'gray'), alpha=0.7)
    
    plt.xlabel('Letter Number')
    plt.ylabel('Strategy Used')
    plt.title('Strategy Evolution')
    plt.yticks([])
    
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=color, label=strategy) 
                      for strategy, color in strategy_colors.items()]
    plt.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.show()

## Exercises <a id='exercises'></a>

### Exercise 1: Production Specificity

Create productions with different levels of specificity and observe which ones win in conflict resolution.

In [ ]:
# Create productions for handling different types of items:
# 1. A general production for any item
# 2. A more specific production for items of a certain category
# 3. A very specific production for a particular item

# Test which production fires for different items

### Exercise 2: Learning Curve

Model how someone learns the multiplication table, starting with counting and progressing to direct retrieval.

In [ ]:
# Create a model with multiple strategies:
# 1. Repeated addition (3×4 = 3+3+3+3)
# 2. Skip counting (3, 6, 9, 12)
# 3. Direct retrieval from memory

# Track how strategy use changes over multiple problems
# Plot the learning curve showing speed improvement

### Exercise 3: Error Productions

Add productions that can make mistakes (like real humans) and productions that detect and correct errors.

In [ ]:
# Create productions that:
# 1. Sometimes make errors (e.g., retrieval failures, wrong calculations)
# 2. Detect when an error might have occurred
# 3. Implement error correction strategies

### Exercise 4: Dual-Task Interference

Model what happens when someone tries to do two tasks that compete for the same cognitive resources.

In [ ]:
# Create two tasks that both need the goal buffer:
# Task 1: Mental arithmetic
# Task 2: Remember a phone number

# Show how performance degrades when trying to do both
# Compare to doing each task alone

## Summary

Productions are ACT-R's mechanism for procedural knowledge. They compete via conflict resolution, improve through utility learning, and model the transition from slow, deliberate performance to fast, automatic execution. The typing model demonstrates how multiple strategies can coexist and shift in dominance as skill develops.

Tutorial 4 will integrate all ACT-R components into complete cognitive models.

## Additional Resources

Research Papers:
- Anderson, J. R. (1993). Rules of the mind.
- Taatgen, N. A., & Lee, F. J. (2003). Production compilation.
- Fu, W. T., & Anderson, J. R. (2006). From recurrent choice to skill learning.

Advanced Topics:
- Production compilation mechanisms
- Utility learning equations
- Partial matching in productions
- Production strength and latency